# 5-Year Treasury Yield Full Refresh Pipeline

## Purpose
This notebook implements a full-refresh ETL pipeline that:
- Fetches complete historical 5-year U.S. Treasury yield data from Alpha Vantage API
- Transforms and standardizes the data
- Overwrites the Delta Lake table with fresh data (full refresh pattern)
- Adds audit columns for data lineage

## Architecture
- **Source**: Alpha Vantage Financial API (CSV format)
- **Target**: Delta table `thekitchen.av.five_year_treasury`
- **Pattern**: Full refresh (overwrite mode) - replaces all data
- **Compute**: Databricks Serverless (Python + PySpark)

## Workflow
1. Define API parameters
2. Fetch complete dataset from API
3. Transform and standardize (pandas → PySpark)
4. Overwrite Delta table (full refresh)
5. Verify results

## Difference from Upsert Pattern
* **Full Refresh**: Replaces entire table (use when API returns complete history)
* **Upsert**: Incremental updates only (use when API returns only new records)

In [0]:
# ==============================================================================
# STEP 1: Alpha Vantage API Parameters Reference
# ==============================================================================
# Documentation for TREASURY_YIELD endpoint configuration
# Alpha Vantage Parameters

# ❚ Required: function
# The function of your choice. In this case, function=TREASURY_YIELD

# ❚ Optional: interval
# By default, interval=monthly. Strings daily, weekly, and monthly are accepted.

# ❚ Optional: maturity
# By default, maturity=10year. Strings 3month, 2year, 5year, 7year, 10year, and 30year are accepted.

# ❚ Optional: datatype
# By default, datatype=json. Strings json and csv are accepted with the following specifications:
# json returns the time series in JSON format;
# csv returns the time series as a CSV (comma separated value) file.

In [0]:
# ==============================================================================
# STEP 2: Configure API request parameters
# ==============================================================================
# API: Alpha Vantage TREASURY_YIELD endpoint
# Returns: Complete historical dataset (not incremental)
# Note: API key should be stored in Databricks secrets for production

import requests

# API key for Alpha Vantage (use dbutils.secrets.get() in production)
api_key = ""

# Define API parameters for 5-year treasury yield
function = "TREASURY_YIELD"  # Treasury yield endpoint
interval = "daily"            # Daily frequency
maturity = "5year"            # 5-year maturity
datatype = "csv"              # CSV format for easy parsing

In [0]:
# ==============================================================================
# STEP 3: Fetch complete historical data from API
# ==============================================================================
# Returns: Full dataset (all historical records)
# Coverage: Historical + latest daily 5-year treasury yields

# Construct the API URL
url = f"https://www.alphavantage.co/query?function={function}&interval={interval}&maturity={maturity}&datatype={datatype}&apikey={api_key}"

# Make the HTTP GET request
r = requests.get(url)
data = r.text

# Preview first 5 lines to verify data format
print("\n".join(data.splitlines()[:5]))

In [0]:
# ==============================================================================
# STEP 4: Transform and full-refresh Delta Lake table
# ==============================================================================
# Process: Pandas (data cleaning) → PySpark (distributed processing)
# Mode: Overwrite (replaces all existing data)

import pandas as pd
import io
from pyspark.sql import functions as F

# Parse CSV string into pandas DataFrame
df = pd.read_csv(io.StringIO(data))

# Convert 'timestamp' column to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.date

# Convert 'value' column to float, coerce errors to NaN
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Pandas → Spark
spark_df = spark.createDataFrame(df)

# Add audit columns in Spark
spark_df = spark_df.withColumn(
    "CreatedAt",
    F.date_trunc(
        "second", F.from_utc_timestamp(F.current_timestamp(), "Europe/Berlin")
    ),
).withColumn("UpdatedAt", F.lit(None).cast("timestamp"))
table_name = "thekitchen.av.five_year_treasury"

# Write to Delta Lake with full refresh (overwrite mode)
try:
    (
        spark_df.write.format("delta")
        .mode("overwrite")  # Full refresh: replaces all data
        .option("mergeSchema", "true")  # Allows schema evolution if needed
        .saveAsTable(table_name)
    )
    print(f"✓ Table '{table_name}' written successfully (full refresh).")
except Exception as e:
    print(f"✗ Failed to write table '{table_name}': {e}")

In [0]:
# ==============================================================================
# UTILITY: Drop table (use only if table structure needs reset)
# ==============================================================================
# Skipped by default - remove %skip to execute

%skip
table_name = "av.five_year_treasury"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

In [0]:
# ==============================================================================
# STEP 5: Verify results
# ==============================================================================
# Display most recent records to confirm successful load

%skip
display(spark_df.orderBy("timestamp", ascending=False).limit(10))

---

## Technical Highlights

### Design Patterns
* **Full Refresh**: Overwrites entire table with complete dataset
* **Simplicity**: No merge logic needed - straightforward ETL
* **Audit Trail**: CreatedAt column for data lineage
* **Schema Evolution**: mergeSchema option handles schema changes

### When to Use Full Refresh vs Upsert
**Full Refresh (this notebook)**:
* API returns complete historical dataset every time
* Table is small enough to rewrite entirely
* Simpler code and easier to maintain
* No need to track incremental changes

**Upsert (other notebooks)**:
* API returns only new/changed records
* Table is large (full rewrite would be expensive)
* Need to preserve historical audit columns
* Incremental pattern is more efficient

### Delta Lake Benefits
* ACID transactions ensure consistency
* Time travel for historical queries
* Overwrite mode is atomic (no partial writes)
* Schema evolution support

### Production Readiness Checklist
- [ ] Store API key in Databricks Secrets (`dbutils.secrets.get()`)
- [ ] Add data validation (row count checks, date range validation)
- [ ] Implement logging (e.g., MLflow or custom logs)
- [ ] Schedule via Databricks Jobs (e.g., daily at market close)
- [ ] Add alerting for API failures or data quality issues
- [ ] Consider partitioning by year for large datasets

### Key Metrics
* **Pattern**: Full table overwrite
* **Data Freshness**: Complete refresh on each run
* **Historical Coverage**: Complete 5-year treasury yield history